# Convert an overwritten customer dim to SCD Type 2 with a MERGE that does not double-insert history

The CRO asked you a question your data cannot answer: "Show me revenue from customers who were in the Enterprise tier 90 days ago, regardless of their tier today." The customer dim is currently overwritten in place; you have zero history. One session to fix the data model so the next version of the question lands cleanly.

The pattern is SCD Type 2 with `valid_from`, `valid_to`, `is_current`. The tool is Athena Iceberg `MERGE INTO`. The gotcha is making it idempotent so the next nightly delta load does not duplicate the history rows you just spent an hour creating.

You have five deliverables:
1. A new `customer_dim_scd2` Iceberg table with the SCD2 column shape, alongside the legacy overwritten `customer_dim_legacy` table seeded with customers.
2. An initial-load MERGE that lifts the legacy dim into `customer_dim_scd2` with every row marked current and the future sentinel on `valid_to`.
3. A delta MERGE that handles insert, update, and the reversion case (a customer goes Pro then Enterprise then Pro again across two deltas).
4. An idempotency proof: re-running the delta MERGE on the same source produces zero new rows.
5. A point-in-time query that returns each customer's tier as of the seed timestamp, regardless of where their tier landed today.

Voice from the brief: this lab is mostly thinking, lightly SQL. Athena scan bytes are tiny; the customer dim is 50 rows, the deltas are three rows, the history table tops out around 53. You will spend more cognitive bandwidth on the reversion case than dollars on the entire lab.

**Time.** Plan for about 75 minutes hands-on. The MERGE statements are short; the thinking is the cost. Session window is 90 minutes.

**Cost.** This lab costs about 5 cents on a clean run and lands around 30 to 60 cents with realistic re-runs and debugging. Athena scans on a 53-row table are well under the smallest billable unit; the floor is the per-query rounded scan minimum. Glue catalog operations are free at this volume. S3 storage on the Iceberg metadata and data files is fractions of a penny. The cleanup cell at the bottom tears down every table, the workgroup, the database, and the bucket so nothing keeps accruing after you finish.

This lab maps to the Data Engineering job-prep track, Warehouse & Lakehouse Mastery sub-track. The MERGE pattern is the canonical dimensional-modeling muscle every analytics-aware data engineering shop expects from a senior.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.8.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import time
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "scd-type-2-from-scratch"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal the brief's slug field exactly
REGION = "us-east-1"  # all labs in this track run in us-east-1

# Resource identifiers. Bucket name is resolved after STS confirms the account.
DB = f"labrun-{LAB_ID}-db"
WG = f"labrun-{LAB_ID}-wg"
TABLE_LEGACY = "customer_dim_legacy"
TABLE_SCD2 = "customer_dim_scd2"
TABLE_DELTA = "customer_source_delta"
BUCKET = None  # set after STS in the credentials cell

# Seed shape: 50 customers, mix of tiers and regions. The Pro/Enterprise/Free
# tier distribution gives the deltas in Task 3 something interesting to do.
SEED_SIZE = 50
TIERS_SEED = ["Free", "Pro", "Enterprise"]
REGIONS_SEED = ["us-east", "us-west", "eu-west", "ap-south"]

# Two customers will be touched by the deltas. The IDs are baked in so the
# checkpoint validators know what to assert without re-deriving them.
CUSTOMER_REVERSION = 7   # starts Pro, goes Enterprise (delta 1), goes Pro again (delta 2)
CUSTOMER_UPDATE = 13     # starts Free, goes Pro (delta 1 only)
CUSTOMER_NEW = 101       # not in seed; inserted by delta 1

# Timestamps drive the SCD2 row boundaries. Fixed so re-runs produce identical
# valid_from / valid_to values and the checkpoint validators can assert exact
# matches. SEED_TS lands the initial load; DELTA_TS_1 and DELTA_TS_2 land the
# two deltas that produce the reversion case.
SEED_TS = "2026-01-01 00:00:00"
DELTA_TS_1 = "2026-02-01 00:00:00"
DELTA_TS_2 = "2026-03-01 00:00:00"
FUTURE_SENTINEL = "9999-12-31 00:00:00"

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All track-data-engineering labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Athena call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that the account ID is in hand. S3 bucket names
# must be globally unique, so we suffix with the account ID.
BUCKET = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in strict reverse-creation order: every
# Iceberg table tears down first (so they release their S3 metadata + data
# objects), then the Athena workgroup, then the Glue database, then the
# bucket. No critical (hourly-billed) resources in this lab; Athena bills
# per-query-scanned only.
#
# Note: labrun-checks 0.8.0 ships AWS adapters for every resource type in
# this manifest, including iceberg_table, athena_workgroup, glue_database,
# and s3_bucket. Iceberg tables drop cleanly via Athena DDL (DROP TABLE
# removes both the Glue catalog entry and the Iceberg manifest pointers).
# The _LabAdapter subclass at the bottom of this notebook predates the
# 0.8.0 iceberg_table adapter and is now functionally equivalent to the
# default; a follow-up can drop it and call run_cleanup against the bundled
# adapter directly. The manifest entries declare the resource type so the
# canonical summary, atexit handler, and orphan scan all see them.
#
# CleanupResource has no `location` field per labrun_checks._types; the
# S3 prefix for each Iceberg table is stashed in the `extra` dict so the
# adapter can empty the prefix if Athena DDL leaves any straggler files.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iceberg_table",
        id=TABLE_SCD2,
        region=REGION,
        parent=DB,
        extra={"location": f"s3://{BUCKET}/{TABLE_SCD2}/", "prefix": f"{TABLE_SCD2}/"},
        cli_delete_command=(
            f"aws athena start-query-execution --query-string "
            f"\"DROP TABLE IF EXISTS {DB}.{TABLE_SCD2}\" "
            f"--work-group {WG} --query-execution-context Database={DB}"
        ),
    ),
    CleanupResource(
        type="iceberg_table",
        id=TABLE_DELTA,
        region=REGION,
        parent=DB,
        extra={"location": f"s3://{BUCKET}/{TABLE_DELTA}/", "prefix": f"{TABLE_DELTA}/"},
        cli_delete_command=(
            f"aws athena start-query-execution --query-string "
            f"\"DROP TABLE IF EXISTS {DB}.{TABLE_DELTA}\" "
            f"--work-group {WG} --query-execution-context Database={DB}"
        ),
    ),
    CleanupResource(
        type="iceberg_table",
        id=TABLE_LEGACY,
        region=REGION,
        parent=DB,
        extra={"location": f"s3://{BUCKET}/{TABLE_LEGACY}/", "prefix": f"{TABLE_LEGACY}/"},
        cli_delete_command=(
            f"aws athena start-query-execution --query-string "
            f"\"DROP TABLE IF EXISTS {DB}.{TABLE_LEGACY}\" "
            f"--work-group {WG} --query-execution-context Database={DB}"
        ),
    ),
    CleanupResource(
        type="athena_workgroup",
        id=WG,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group --work-group {WG} "
            f"--recursive-delete-option"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DB}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell at the bottom is the authoritative entry
    point; this is the safety net for kernel crashes. Errors are swallowed
    because atexit handlers must not raise.
    """
    try:
        if "_LabAdapter" in globals():
            run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())
        else:
            run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources. This prevents duplicate-bucket and duplicate-database
    scenarios that produce confusing AlreadyExists errors mid-lab.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can collide on the bucket name,")
    print("the Glue database, or the Athena workgroup. Run the cleanup cell")
    print("at the bottom of this notebook first, or remove the resources")
    print("manually with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

In [ ]:
# NBVAL_SKIP
# Shared infrastructure: S3 bucket, Glue database, Athena workgroup with
# Engine v3 (Iceberg MERGE syntax depends on v3; v2 is the silent
# default-rollover landmine the brief calls out). Also defines the
# run_athena helper every task cell uses to submit DDL/DML and poll until
# the query terminates.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Bucket. us-east-1 rejects LocationConstraint; outside us-east-1 you would
# pass CreateBucketConfiguration={"LocationConstraint": REGION}.
try:
    s3.create_bucket(Bucket=BUCKET)
    s3.put_bucket_tagging(
        Bucket=BUCKET,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Bucket created and tagged: {BUCKET}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou",):
        print(f"Bucket {BUCKET} already owned by this account, reusing.")
    else:
        raise

# Glue database.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DB,
            "Description": f"labrun {LAB_ID} database",
        },
    )
    # Glue databases do not accept tags inline. Tag the catalog database via
    # the resource-tagging API so the orphan scan finds it.
    db_arn = f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DB}"
    glue.tag_resource(
        ResourceArn=db_arn,
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DB}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DB} already exists, reusing.")
    else:
        raise

# Athena workgroup. Engine v3 is pinned explicitly. Iceberg MERGE syntax
# requires Engine v3; Engine v2 fails the MERGE with a confusing parse
# error that does not mention Iceberg. The workgroup also pins the result
# location so every query in the lab writes to the same prefix.
athena_results_uri = f"s3://{BUCKET}/athena-results/"
try:
    athena.create_work_group(
        Name=WG,
        Configuration={
            "ResultConfiguration": {"OutputLocation": athena_results_uri},
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
            "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
        },
        Description=f"Lab workgroup for {LAB_ID}.",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Athena workgroup created on Engine v3: {WG}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code == "InvalidRequestException":
        # Already exists. Force Engine v3 on the existing workgroup in case a
        # prior run created it on v2.
        athena.update_work_group(
            WorkGroup=WG,
            ConfigurationUpdates={
                "ResultConfigurationUpdates": {"OutputLocation": athena_results_uri},
                "EnforceWorkGroupConfiguration": True,
                "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            },
            State="ENABLED",
        )
        print(f"Athena workgroup {WG} already exists, reused on Engine v3.")
    else:
        raise


def run_athena(sql: str, timeout_s: int = 180) -> dict:
    """Submit one query against the lab workgroup and poll until terminal.

    Returns the full QueryExecution dict on SUCCEEDED. Raises RuntimeError
    with the StateChangeReason on FAILED/CANCELLED so the caller does not
    silently treat a failed MERGE as a success.
    """
    start = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DB},
        WorkGroup=WG,
    )
    qid = start["QueryExecutionId"]
    deadline = time.time() + timeout_s
    waits = [
        "Athena is committing the Iceberg snapshot...",
        "Counting your historical rows. The merge takes a moment...",
        "The reversion case is making Athena think harder than usual...",
    ]
    wait_idx = 0
    while time.time() < deadline:
        resp = athena.get_query_execution(QueryExecutionId=qid)
        state = resp["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            return resp["QueryExecution"]
        if state in ("FAILED", "CANCELLED"):
            reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "")
            raise RuntimeError(f"Athena query {state}: {reason}\n  SQL: {sql[:200]}")
        if wait_idx < len(waits):
            print(waits[wait_idx])
            wait_idx += 1
        time.sleep(2)
    raise RuntimeError(f"Athena query {qid} did not finish within {timeout_s} seconds.")


def fetch_scalar(sql: str):
    """Run a single-row, single-column query and return the typed value."""
    execution = run_athena(sql)
    qid = execution["QueryExecutionId"]
    rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    if len(rows) < 2:
        return None
    data = rows[1]["Data"][0]
    return data.get("VarCharValue")


print()
print("Shared infrastructure ready: bucket, database, workgroup (Engine v3).")

## Task 1: Stand up the SCD2 target table and the legacy source you have to migrate from

Two Iceberg tables before any MERGE happens.

First, the legacy overwritten dim. `customer_dim_legacy` holds one row per customer. The CRO's question cannot be answered from this table because every nightly load overwrites it. You will seed it with 50 customers via `INSERT INTO ... VALUES`.

Second, the SCD2 target. `customer_dim_scd2` shares the business columns (`customer_id`, `tier`, `region`) and adds three temporal columns: `valid_from TIMESTAMP`, `valid_to TIMESTAMP`, `is_current BOOLEAN`. The future sentinel `TIMESTAMP '9999-12-31 00:00:00'` is what we will write to `valid_to` for any currently-active row; a NULL sentinel breaks `BETWEEN` in the Task 5 point-in-time query and is the wrong shape to live with.

Build it in this order:
1. `CREATE TABLE IF NOT EXISTS` for `customer_dim_legacy` with three columns (`customer_id`, `tier`, `region`) and `TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')`. Set `location` to `s3://{BUCKET}/customer_dim_legacy/`.
2. `CREATE TABLE IF NOT EXISTS` for `customer_dim_scd2` with the three business columns plus `valid_from TIMESTAMP`, `valid_to TIMESTAMP`, `is_current BOOLEAN`. Same Iceberg table properties.
3. Seed `customer_dim_legacy` with the 50-row `INSERT INTO ... VALUES` payload constructed in the cell. The SCD2 table stays empty in this task; Task 2 fills it via the initial-load MERGE.

The seed payload is deterministic: customer ids 1 through 50, tier cycles through `Free / Pro / Enterprise`, region cycles through the four regions. Customer 7 lands on `Pro`; customer 13 lands on `Free`. The deltas in Task 3 will move both.

In [ ]:
# NBVAL_SKIP
# Task 1: create both Iceberg tables, then seed customer_dim_legacy with 50
# deterministic rows. The seed payload is built in Python so the row values
# match what Task 5's point-in-time validator expects.

# Build the 50-row seed payload deterministically. Cycling through the tier
# and region lists produces the same rows on every re-run; customer 7 lands
# on Pro (used by the reversion delta) and customer 13 lands on Free (used
# by the update delta).
seed_rows = []
for cid in range(1, SEED_SIZE + 1):
    tier = TIERS_SEED[cid % len(TIERS_SEED)]
    region = REGIONS_SEED[cid % len(REGIONS_SEED)]
    seed_rows.append((cid, tier, region))

values_clause = ", ".join(
    f"({cid}, '{tier}', '{region}')" for cid, tier, region in seed_rows
)

# Two Iceberg DDLs and one INSERT. Each statement runs through run_athena
# which polls the QueryExecution and raises on FAILED so a silent typo
# surfaces immediately.

# YOUR CODE: Run a CREATE TABLE IF NOT EXISTS for customer_dim_legacy on
# Iceberg. Columns: customer_id int, tier string, region string.
# LOCATION 's3://{BUCKET}/{TABLE_LEGACY}/' and
# TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet'). Use the
# run_athena(sql) helper from the previous cell.
print(f"Created Iceberg table {DB}.{TABLE_LEGACY}")

# YOUR CODE: Run a CREATE TABLE IF NOT EXISTS for customer_dim_scd2 on
# Iceberg. Columns: customer_id int, tier string, region string,
# valid_from timestamp, valid_to timestamp, is_current boolean.
# LOCATION 's3://{BUCKET}/{TABLE_SCD2}/' and the same TBLPROPERTIES.
print(f"Created Iceberg table {DB}.{TABLE_SCD2}")

# YOUR CODE: Run a single INSERT INTO {DB}.{TABLE_LEGACY} (customer_id, tier,
# region) VALUES <values_clause>. The values_clause string is already built.
print(f"Seeded {DB}.{TABLE_LEGACY} with {SEED_SIZE} rows.")

# Quick confirmation print without burning another Athena query: the seed
# rows are deterministic, so we already know the shape.
print(f"  Customer {CUSTOMER_REVERSION} seeded as: tier=Pro (will move Pro -> Enterprise -> Pro)")
print(f"  Customer {CUSTOMER_UPDATE} seeded as: tier=Free (will move Free -> Pro)")
print(f"  Customer {CUSTOMER_NEW} not in seed (will be inserted by delta 1)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: both Iceberg tables exist in Glue with the right columns,
# and customer_dim_legacy contains exactly SEED_SIZE rows.

def checkpoint_1(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Both tables must exist in the catalog.
        try:
            legacy_meta = glue_client.get_table(DatabaseName=DB, Name=TABLE_LEGACY)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue table {DB}.{TABLE_LEGACY} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            scd2_meta = glue_client.get_table(DatabaseName=DB, Name=TABLE_SCD2)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue table {DB}.{TABLE_SCD2} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        # SCD2 table must have the six expected columns.
        scd2_cols = {
            c["Name"]: c["Type"]
            for c in scd2_meta["Table"]["StorageDescriptor"].get("Columns", [])
        }
        required_scd2 = {
            "customer_id": "int",
            "tier": "string",
            "region": "string",
            "valid_from": "timestamp",
            "valid_to": "timestamp",
            "is_current": "boolean",
        }
        missing = [
            f"{name} {required_type}"
            for name, required_type in required_scd2.items()
            if name not in scd2_cols
        ]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {DB}.{TABLE_SCD2} is missing columns: "
                    f"{', '.join(missing)}. Found: {sorted(scd2_cols.keys())}."
                ),
            )

        # Legacy table row count via Athena.
        count_str = fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_LEGACY}")
        try:
            count = int(count_str) if count_str is not None else 0
        except ValueError:
            return CheckpointResult(
                status="error",
                error_reason=f"Could not parse legacy count: {count_str!r}",
            )
        if count != SEED_SIZE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected {DB}.{TABLE_LEGACY} to have {SEED_SIZE} rows, "
                    f"found {count}. Re-run the seed INSERT."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reports either a missing table, missing columns, or a wrong row count. Read the failure reason. Iceberg DDL is the same as plain Athena DDL with two additions: the `LOCATION` clause (an S3 prefix the table will own) and a `TBLPROPERTIES` clause that names the table type.

</details>

<details><summary>Hint 2 (direction)</summary>

Iceberg tables in Athena need `TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')` and a `LOCATION 's3://bucket/path/'` clause. Column types: `customer_id int`, `tier string`, `region string`, `valid_from timestamp`, `valid_to timestamp`, `is_current boolean`. The seed INSERT is a single statement with all 50 row tuples in a comma-separated `VALUES` payload; the `values_clause` Python string is already built in the cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
run_athena(f"""
CREATE TABLE IF NOT EXISTS {DB}.{TABLE_LEGACY} (
  customer_id int,
  tier string,
  region string
)
LOCATION 's3://{BUCKET}/{TABLE_LEGACY}/'
TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')
""")
print(f"Created Iceberg table {DB}.{TABLE_LEGACY}")

run_athena(f"""
CREATE TABLE IF NOT EXISTS {DB}.{TABLE_SCD2} (
  customer_id int,
  tier string,
  region string,
  valid_from timestamp,
  valid_to timestamp,
  is_current boolean
)
LOCATION 's3://{BUCKET}/{TABLE_SCD2}/'
TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')
""")
print(f"Created Iceberg table {DB}.{TABLE_SCD2}")

run_athena(
    f"INSERT INTO {DB}.{TABLE_LEGACY} (customer_id, tier, region) VALUES {values_clause}"
)
print(f"Seeded {DB}.{TABLE_LEGACY} with {SEED_SIZE} rows.")
```

If the CREATE returns `EXTERNAL_LOCATION_NOT_EMPTY`, an Iceberg table previously lived at that S3 prefix and was dropped without clearing the metadata. Run the cleanup cell at the bottom of the notebook to empty the bucket, then re-run setup. If the seed INSERT fails with a parse error, the most common cause is a tier or region value with a single quote in it; the deterministic generator avoids that but a custom override could trigger it.

</details>

**Wallet check.** About one penny so far. Three Athena queries (two DDL plus the seed INSERT) at the per-query rounded minimum scan, S3 PutObject calls for the Iceberg metadata files. Your morning coffee cost two hundred times more.

## Task 2: Initial load. Lift the legacy dim into SCD2 with the future sentinel on every row

`customer_dim_scd2` is empty. Every row in `customer_dim_legacy` represents a customer who has been in their current tier since `SEED_TS` (the moment you took over the data model). For the initial load, every legacy row becomes one SCD2 row with `valid_from = SEED_TS`, `valid_to = TIMESTAMP '9999-12-31 00:00:00'`, `is_current = true`.

The MERGE in this task is the simplest in the lab; there is nothing to update, only to insert. After the MERGE:
- `count(*) FROM customer_dim_scd2` equals `count(*) FROM customer_dim_legacy` (50).
- Every row in `customer_dim_scd2` has `is_current = true`.
- Every `valid_from` is `SEED_TS`, every `valid_to` is the future sentinel.

The MERGE shape: `MERGE INTO target USING (SELECT customer_id, tier, region FROM legacy) source ON target.customer_id = source.customer_id WHEN NOT MATCHED THEN INSERT VALUES (...)`. No MATCHED branch is needed because the target is empty on the first run; the predicate keeps the statement idempotent (re-running it after the table is populated does nothing). The sentinel timestamp goes in as `TIMESTAMP '9999-12-31 00:00:00'`.

In [ ]:
# NBVAL_SKIP
# Task 2: initial-load MERGE. Lift every legacy row into customer_dim_scd2
# as a current row. The MERGE predicate makes this idempotent: re-running
# the statement after the table is populated will match every row on the
# customer_id key and do nothing because no MATCHED branch is provided.

# YOUR CODE: Run a MERGE statement that pulls customer_id, tier, region from
# {DB}.{TABLE_LEGACY} (aliased source), matches on target.customer_id =
# source.customer_id, and on WHEN NOT MATCHED inserts (source.customer_id,
# source.tier, source.region, TIMESTAMP '{SEED_TS}', TIMESTAMP
# '{FUTURE_SENTINEL}', true). Use the run_athena helper. The target table is
# {DB}.{TABLE_SCD2}.

print(f"Initial-load MERGE complete. {TABLE_SCD2} now mirrors {TABLE_LEGACY} at SEED_TS={SEED_TS}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: customer_dim_scd2 has the same current-row count as the
# legacy table; every row has valid_from = SEED_TS, valid_to = future
# sentinel, is_current = true.

def checkpoint_2(session):
    try:
        # Current count must match legacy count.
        legacy_count = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_LEGACY}") or 0)
        current_count = int(fetch_scalar(
            f"SELECT count(*) FROM {DB}.{TABLE_SCD2} WHERE is_current = true"
        ) or 0)
        if current_count != legacy_count:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Current-row count in {TABLE_SCD2} is {current_count}, "
                    f"expected {legacy_count} (matching {TABLE_LEGACY}). "
                    f"The initial-load MERGE did not insert every legacy row."
                ),
            )

        # Every current row must have valid_from = SEED_TS, valid_to = sentinel.
        bad = int(fetch_scalar(
            f"SELECT count(*) FROM {DB}.{TABLE_SCD2} "
            f"WHERE is_current = true "
            f"AND (valid_from <> TIMESTAMP '{SEED_TS}' "
            f"OR valid_to <> TIMESTAMP '{FUTURE_SENTINEL}')"
        ) or 0)
        if bad > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{bad} current rows in {TABLE_SCD2} have wrong "
                    f"valid_from or valid_to. Expected valid_from='{SEED_TS}' "
                    f"and valid_to='{FUTURE_SENTINEL}' on every current row."
                ),
            )

        # Row-level diff: every legacy customer_id must appear as a current
        # row in SCD2 with the same tier and region.
        diff = int(fetch_scalar(f"""
            SELECT count(*) FROM (
                SELECT customer_id, tier, region FROM {DB}.{TABLE_LEGACY}
                EXCEPT
                SELECT customer_id, tier, region FROM {DB}.{TABLE_SCD2}
                WHERE is_current = true
            )
        """) or 0)
        if diff > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{diff} legacy rows have no matching current row in "
                    f"{TABLE_SCD2}. Check that the MERGE pulled customer_id, "
                    f"tier, and region from the legacy table verbatim."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three properties: current-row count matches the legacy count, every current row has the right `valid_from` and `valid_to` literals, and the legacy rows project cleanly into the current-row slice. Read the failure reason. The most common miss on the first run is forgetting `TIMESTAMP` in front of the date literal; Athena will parse the row but the comparison in the checkpoint fails because string `'9999-12-31'` is not a timestamp value.

</details>

<details><summary>Hint 2 (direction)</summary>

The MERGE shape is one MATCHED-less branch: `MERGE INTO target USING (SELECT customer_id, tier, region FROM legacy) source ON target.customer_id = source.customer_id WHEN NOT MATCHED THEN INSERT (...) VALUES (source.customer_id, source.tier, source.region, TIMESTAMP '{SEED_TS}', TIMESTAMP '{FUTURE_SENTINEL}', true)`. No WHEN MATCHED branch is required; the target is empty on the first run, and the ON predicate makes the statement idempotent.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
run_athena(f"""
MERGE INTO {DB}.{TABLE_SCD2} target
USING (
  SELECT customer_id, tier, region FROM {DB}.{TABLE_LEGACY}
) source
ON target.customer_id = source.customer_id
WHEN NOT MATCHED THEN
  INSERT (customer_id, tier, region, valid_from, valid_to, is_current)
  VALUES (
    source.customer_id,
    source.tier,
    source.region,
    TIMESTAMP '{SEED_TS}',
    TIMESTAMP '{FUTURE_SENTINEL}',
    true
  )
""")
print(f"Initial-load MERGE complete. {TABLE_SCD2} now mirrors {TABLE_LEGACY} at SEED_TS={SEED_TS}.")
```

If the MERGE fails with `MERGE INTO is only supported for ACID tables` or a parse error, the workgroup is on Engine v2. The setup cell pins Engine v3; if you see this error, re-run the setup infrastructure cell to force the workgroup to v3.

</details>

**Wallet check.** About two cents so far. The MERGE counts as one Athena query, plus the checkpoint runs four scalar queries. Iceberg metadata writes add a few S3 PutObject calls. Still rounding to zero on any honest pricing calculator.

## Task 3: Apply two deltas. Insert one customer, update one, reverse one

Here is the gotcha at the heart of the lab. SCD Type 2 tutorials usually show one delta with one INSERT and one UPDATE and call it done. The reversion case is the one that trips most engineers: a customer goes Pro then Enterprise then Pro again across two nightly loads. Customer 7 (seeded as Pro) gets a delta to Enterprise tonight (delta 1), then a delta back to Pro tomorrow night (delta 2). The SCD2 table should end up with THREE rows for customer 7: the original Pro row closed at delta 1's timestamp, the Enterprise row closed at delta 2's timestamp, and the current Pro row open with the future sentinel.

The Athena Iceberg MERGE constraint that drives this task: a single MERGE statement cannot both close a current row AND insert a new current row for the same source row. The supported pattern is two statements:
1. **MERGE** closes any current row whose tier no longer matches the source. The MATCHED branch sets `is_current = false` and `valid_to = <delta_ts>`. The NOT MATCHED branch handles brand-new customers (insert current row with the future sentinel).
2. **INSERT** lands the new current row for every source row whose tier changed. After step 1 closed the prior current row, step 2 inserts the replacement current row.

We apply this pattern twice: once for delta 1 (Pro -> Enterprise for customer 7, Free -> Pro for customer 13, new customer 101 at Free), once for delta 2 (Enterprise -> Pro for customer 7). After both deltas:
- Customer 7: 3 rows. (original Pro closed at DELTA_TS_1, Enterprise closed at DELTA_TS_2, Pro current)
- Customer 13: 2 rows. (Free closed at DELTA_TS_1, Pro current)
- Customer 101: 1 row. (Free current)
- Other 48 customers: 1 row each, unchanged.

Total: 48 + 3 + 2 + 1 = 53 rows.

In [ ]:
# NBVAL_SKIP
# Task 3: apply delta 1 and delta 2. Each delta is a two-statement pattern:
# MERGE closes any current row whose tier no longer matches the source
# (MATCHED branch) and inserts new customers (NOT MATCHED branch). A
# follow-up INSERT lands the replacement current row for customers whose
# tier changed.
#
# A source delta table is materialized once per delta and dropped, so each
# delta is structurally identical. The customer_source_delta table is
# declared in the cleanup manifest because the lab persists it for the
# duration; we replace its contents between deltas via DELETE + INSERT.

# Create the source-delta table once (idempotent on re-run).
run_athena(f"""
CREATE TABLE IF NOT EXISTS {DB}.{TABLE_DELTA} (
  customer_id int,
  tier string,
  region string
)
LOCATION 's3://{BUCKET}/{TABLE_DELTA}/'
TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')
""")
print(f"Source-delta Iceberg table ready: {DB}.{TABLE_DELTA}")


def apply_delta(delta_rows: list[tuple], delta_ts: str, label: str) -> None:
    """Replace the source-delta table contents with delta_rows, then MERGE
    + INSERT into the SCD2 target. The MERGE closes any current row whose
    tier no longer matches; the INSERT lands the new current row.
    """
    print(f"Applying {label} at {delta_ts} ({len(delta_rows)} rows)...")

    # Clear the delta table for this round.
    run_athena(f"DELETE FROM {DB}.{TABLE_DELTA}")
    values_clause_d = ", ".join(
        f"({cid}, '{tier}', '{region}')" for cid, tier, region in delta_rows
    )
    run_athena(
        f"INSERT INTO {DB}.{TABLE_DELTA} (customer_id, tier, region) "
        f"VALUES {values_clause_d}"
    )

    # YOUR CODE: Step 1 of the SCD2 delta pattern. Run a MERGE INTO
    # {DB}.{TABLE_SCD2} target USING {DB}.{TABLE_DELTA} source ON
    # target.customer_id = source.customer_id AND target.is_current = true.
    # WHEN MATCHED AND source.tier <> target.tier THEN UPDATE SET
    # valid_to = TIMESTAMP '{delta_ts}', is_current = false.
    # WHEN NOT MATCHED THEN INSERT (customer_id, tier, region, valid_from,
    # valid_to, is_current) VALUES (source.customer_id, source.tier,
    # source.region, TIMESTAMP '{delta_ts}', TIMESTAMP '{FUTURE_SENTINEL}',
    # true). The MATCHED-AND-tier-changed predicate is what makes this
    # idempotent in Task 4; if you drop the predicate the second run will
    # close every current row a second time.

    # YOUR CODE: Step 2 of the SCD2 delta pattern. Run an INSERT INTO
    # {DB}.{TABLE_SCD2} (customer_id, tier, region, valid_from, valid_to,
    # is_current) SELECT s.customer_id, s.tier, s.region, TIMESTAMP
    # '{delta_ts}', TIMESTAMP '{FUTURE_SENTINEL}', true FROM
    # {DB}.{TABLE_DELTA} s WHERE EXISTS (SELECT 1 FROM {DB}.{TABLE_SCD2} t
    # WHERE t.customer_id = s.customer_id AND t.is_current = false AND
    # t.valid_to = TIMESTAMP '{delta_ts}'). The WHERE-EXISTS subquery picks
    # up exactly the rows that step 1 just closed (i.e., customers whose
    # tier just changed); brand-new customers were already inserted by the
    # NOT MATCHED branch of step 1.

    print(f"  {label} applied.")


# Delta 1: customer 7 Pro -> Enterprise, customer 13 Free -> Pro, new customer 101 at Free.
delta_1 = [
    (CUSTOMER_REVERSION, "Enterprise", REGIONS_SEED[CUSTOMER_REVERSION % len(REGIONS_SEED)]),
    (CUSTOMER_UPDATE, "Pro", REGIONS_SEED[CUSTOMER_UPDATE % len(REGIONS_SEED)]),
    (CUSTOMER_NEW, "Free", REGIONS_SEED[CUSTOMER_NEW % len(REGIONS_SEED)]),
]
apply_delta(delta_1, DELTA_TS_1, "delta 1")

# Delta 2: customer 7 Enterprise -> Pro (the reversion). Customer 13 stays at
# Pro (a no-op against the SCD2 target because the MATCHED-AND-tier-changed
# predicate sees source.tier == target.tier). Customer 101 stays at Free.
delta_2 = [
    (CUSTOMER_REVERSION, "Pro", REGIONS_SEED[CUSTOMER_REVERSION % len(REGIONS_SEED)]),
    (CUSTOMER_UPDATE, "Pro", REGIONS_SEED[CUSTOMER_UPDATE % len(REGIONS_SEED)]),
    (CUSTOMER_NEW, "Free", REGIONS_SEED[CUSTOMER_NEW % len(REGIONS_SEED)]),
]
apply_delta(delta_2, DELTA_TS_2, "delta 2 (reversion)")

print()
print("Both deltas applied. The SCD2 history should now hold:")
print(f"  customer {CUSTOMER_REVERSION}: 3 rows (Pro closed at {DELTA_TS_1}, Enterprise closed at {DELTA_TS_2}, Pro current)")
print(f"  customer {CUSTOMER_UPDATE}: 2 rows (Free closed at {DELTA_TS_1}, Pro current)")
print(f"  customer {CUSTOMER_NEW}: 1 row (Free current)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: after both deltas, customer_dim_scd2 has the right total
# row count and the right per-customer shape for the three touched
# customers.

def checkpoint_3(session):
    try:
        # Total rows after both deltas: 48 unchanged + 3 (reversion) + 2 (update) + 1 (new) = 54.
        # Wait: 50 seed - 2 touched seed rows = 48 untouched. Customer 7
        # contributes 3 rows total, customer 13 contributes 2, customer 101
        # contributes 1 (new). 48 + 3 + 2 + 1 = 54... Recount: of the 50
        # seed customers, 7 and 13 are touched, so 48 untouched seed rows.
        # Customer 7 ends with 3 rows in SCD2. Customer 13 ends with 2.
        # Customer 101 ends with 1. 48 + 3 + 2 + 1 = 54.
        expected_total = (SEED_SIZE - 2) + 3 + 2 + 1
        total = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_SCD2}") or 0)
        if total != expected_total:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total rows in {TABLE_SCD2} is {total}, expected {expected_total}. "
                    f"Either a delta inserted the wrong number of rows or the MATCHED "
                    f"predicate closed rows that should have stayed current."
                ),
            )

        # Per-customer row counts.
        rev_count = int(fetch_scalar(
            f"SELECT count(*) FROM {DB}.{TABLE_SCD2} WHERE customer_id = {CUSTOMER_REVERSION}"
        ) or 0)
        if rev_count != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Customer {CUSTOMER_REVERSION} (reversion case) has {rev_count} rows "
                    f"in {TABLE_SCD2}, expected 3 (Pro closed at delta 1, Enterprise closed "
                    f"at delta 2, Pro current). The two-statement MERGE+INSERT pattern is "
                    f"how the reversion lands; a single MERGE cannot both close and insert."
                ),
            )

        upd_count = int(fetch_scalar(
            f"SELECT count(*) FROM {DB}.{TABLE_SCD2} WHERE customer_id = {CUSTOMER_UPDATE}"
        ) or 0)
        if upd_count != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Customer {CUSTOMER_UPDATE} has {upd_count} rows in {TABLE_SCD2}, "
                    f"expected 2 (Free closed at delta 1, Pro current)."
                ),
            )

        new_count = int(fetch_scalar(
            f"SELECT count(*) FROM {DB}.{TABLE_SCD2} WHERE customer_id = {CUSTOMER_NEW}"
        ) or 0)
        if new_count != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"New customer {CUSTOMER_NEW} has {new_count} rows in {TABLE_SCD2}, "
                    f"expected exactly 1 current row. The NOT MATCHED branch of the MERGE "
                    f"is what inserts brand-new customers."
                ),
            )

        # Reversion customer must end on tier = Pro and is_current = true.
        current_tier_rev = fetch_scalar(
            f"SELECT tier FROM {DB}.{TABLE_SCD2} "
            f"WHERE customer_id = {CUSTOMER_REVERSION} AND is_current = true"
        )
        if current_tier_rev != "Pro":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Customer {CUSTOMER_REVERSION} current-row tier is {current_tier_rev!r}, "
                    f"expected 'Pro' after the delta-2 reversion. The delta-2 INSERT did "
                    f"not land or the MERGE failed to close the Enterprise row."
                ),
            )

        # Exactly one current row per customer; no double-currents (the
        # classic double-insert bug shows up here).
        double_currents = int(fetch_scalar(f"""
            SELECT count(*) FROM (
                SELECT customer_id, count(*) AS c FROM {DB}.{TABLE_SCD2}
                WHERE is_current = true
                GROUP BY customer_id
                HAVING count(*) > 1
            )
        """) or 0)
        if double_currents > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{double_currents} customers have more than one current row in "
                    f"{TABLE_SCD2}. The classic double-insert bug; check that the MERGE "
                    f"closes the prior current row before the INSERT lands the new one."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure modes dominate this checkpoint. Either the reversion customer has 2 rows instead of 3 (you treated the second tier change as if it were the same customer's first change), or some customer has 2 current rows (the classic double-insert: you ran INSERT without first closing the prior current row, or you closed and inserted in the same MERGE statement and the planner double-counted).

</details>

<details><summary>Hint 2 (direction)</summary>

Athena Iceberg MERGE does not let you both close a current row and insert a new current row for the same source row in one statement. Use two statements per delta: a MERGE that closes any matched-current-row whose tier no longer equals the source tier (MATCHED branch sets `is_current=false` and `valid_to=<delta_ts>`) and inserts brand-new customers (NOT MATCHED branch with `valid_to=<future sentinel>` and `is_current=true`), then an INSERT that lands the replacement current row for any source row whose tier just changed (use a WHERE EXISTS subquery against the rows you just closed). Run the two-statement pattern once per delta, twice total for the reversion.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for `apply_delta`:

```python
def apply_delta(delta_rows, delta_ts, label):
    print(f"Applying {label} at {delta_ts} ({len(delta_rows)} rows)...")

    run_athena(f"DELETE FROM {DB}.{TABLE_DELTA}")
    values_clause_d = ", ".join(
        f"({cid}, '{tier}', '{region}')" for cid, tier, region in delta_rows
    )
    run_athena(
        f"INSERT INTO {DB}.{TABLE_DELTA} (customer_id, tier, region) "
        f"VALUES {values_clause_d}"
    )

    # Step 1: MERGE closes any current row whose tier no longer matches and
    # inserts brand-new customers via the NOT MATCHED branch.
    run_athena(f"""
    MERGE INTO {DB}.{TABLE_SCD2} target
    USING {DB}.{TABLE_DELTA} source
    ON target.customer_id = source.customer_id
       AND target.is_current = true
    WHEN MATCHED AND source.tier <> target.tier THEN
      UPDATE SET valid_to = TIMESTAMP '{delta_ts}', is_current = false
    WHEN NOT MATCHED THEN
      INSERT (customer_id, tier, region, valid_from, valid_to, is_current)
      VALUES (
        source.customer_id, source.tier, source.region,
        TIMESTAMP '{delta_ts}', TIMESTAMP '{FUTURE_SENTINEL}', true
      )
    """)

    # Step 2: INSERT lands the replacement current row for any source row
    # whose tier just changed (i.e., the row step 1 just closed exists).
    run_athena(f"""
    INSERT INTO {DB}.{TABLE_SCD2}
    (customer_id, tier, region, valid_from, valid_to, is_current)
    SELECT s.customer_id, s.tier, s.region,
           TIMESTAMP '{delta_ts}', TIMESTAMP '{FUTURE_SENTINEL}', true
    FROM {DB}.{TABLE_DELTA} s
    WHERE EXISTS (
      SELECT 1 FROM {DB}.{TABLE_SCD2} t
      WHERE t.customer_id = s.customer_id
        AND t.is_current = false
        AND t.valid_to = TIMESTAMP '{delta_ts}'
    )
    """)

    print(f"  {label} applied.")
```

The `MATCHED AND source.tier <> target.tier` predicate is what makes the pattern idempotent against Task 4: re-running on the same delta sees `source.tier == target.tier` for the now-current row and no-ops. The WHERE EXISTS subquery in step 2 ties the INSERT to exactly the rows step 1 just closed, so brand-new customers (handled by the NOT MATCHED branch) are not double-inserted.

</details>

**Wallet check.** About four cents so far. Two deltas, each with four Athena queries (DELETE, INSERT, MERGE, INSERT) plus the source-delta CREATE and the checkpoint verification queries. Still rounds to pocket change.

## Task 4: Idempotency. Re-running the delta MERGE on the same source produces zero new rows

An ETL job that doubles your history table on the second run is worse than no SCD2 at all. Idempotency means: capture the row count, re-run the delta pipeline on the SAME source delta, capture the row count again; the difference is zero.

The `MATCHED AND source.tier <> target.tier` predicate from Task 3 is the load-bearing piece. Without it, the second MERGE run re-closes every current row a second time and the INSERT lands a duplicate current row. With it, the second run sees `source.tier == target.tier` for every row and the MATCHED branch no-ops; the WHERE EXISTS subquery in step 2 then finds no newly-closed rows and the INSERT no-ops too.

Re-apply delta 2 (still in `customer_source_delta`) and assert the row count is unchanged.

If your Task 3 MERGE was missing the `AND source.tier <> target.tier` predicate, this checkpoint is where you find out.

In [ ]:
# NBVAL_SKIP
# Task 4: re-apply delta 2 and assert the row count is unchanged.
# delta_2 is still in scope from Task 3; we re-apply it through the same
# apply_delta helper, which guarantees the second run is structurally
# identical to the first.

before = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_SCD2}") or 0)
print(f"Row count before re-running delta 2: {before}")

# YOUR CODE: Call apply_delta(delta_2, DELTA_TS_2, "delta 2 re-run") to
# re-apply the same delta against the SCD2 target. If the MERGE predicate
# from Task 3 includes AND source.tier <> target.tier, the re-run no-ops.
# If not, the row count after will exceed before.

after = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_SCD2}") or 0)
print(f"Row count after re-running delta 2: {after}")
print(f"Delta: {after - before}")
if after == before:
    print("PASS: re-running delta 2 was a no-op. Idempotency holds.")
else:
    print("FAIL: row count changed on the re-run. Check the MATCHED predicate in your MERGE.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: independent idempotency verification. Capture the row count,
# re-apply delta 2 one more time, capture again, and assert the delta is
# zero. The checkpoint runs the re-apply itself so it does not depend on
# whether the student ran the task cell correctly.

def checkpoint_4(session):
    try:
        before_c = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_SCD2}") or 0)

        # Re-apply delta 2 directly so the checkpoint is self-contained.
        # Same shape as the task cell: clear customer_source_delta, insert
        # the three delta-2 rows, run the MERGE + INSERT pattern.
        run_athena(f"DELETE FROM {DB}.{TABLE_DELTA}")
        values_clause_d = ", ".join(
            f"({cid}, '{tier}', '{region}')" for cid, tier, region in delta_2
        )
        run_athena(
            f"INSERT INTO {DB}.{TABLE_DELTA} (customer_id, tier, region) "
            f"VALUES {values_clause_d}"
        )
        # The MERGE and follow-up INSERT are whatever the student wrote in
        # apply_delta; calling apply_delta exercises that code path.
        apply_delta(delta_2, DELTA_TS_2, "delta 2 checkpoint re-run")

        after_c = int(fetch_scalar(f"SELECT count(*) FROM {DB}.{TABLE_SCD2}") or 0)
        diff = after_c - before_c
        if diff != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Re-running delta 2 changed the row count by {diff} "
                    f"({before_c} before, {after_c} after). The MERGE is not "
                    f"idempotent. Add `AND source.tier <> target.tier` to the "
                    f"WHEN MATCHED branch so already-applied deltas no-op."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the row count moved on the re-run, the MERGE is closing rows that should not be closed. The MATCHED branch is firing on every row that matches `customer_id` regardless of whether the tier actually changed. There is a predicate missing from the MATCHED branch.

</details>

<details><summary>Hint 2 (direction)</summary>

The fix is in the Task 3 MERGE, not in this task's cell. Find your MERGE statement and add `AND source.tier <> target.tier` to the WHEN MATCHED clause. After the fix, the MATCHED branch only fires when the source tier differs from the current row's tier; on a re-run, every source tier matches the now-current row's tier and the branch no-ops. The WHERE EXISTS subquery in step 2 then finds no newly-closed rows and the INSERT no-ops too.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The MERGE statement in Task 3's `apply_delta` helper should look like this (note the `AND source.tier <> target.tier` clause):

```sql
MERGE INTO target
USING source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED AND source.tier <> target.tier THEN
  UPDATE SET valid_to = TIMESTAMP '...', is_current = false
WHEN NOT MATCHED THEN
  INSERT (...) VALUES (...)
```

If the MERGE is already correct and the row count still moves on the re-run, the second statement (the INSERT) is the culprit; its WHERE EXISTS subquery should filter on `t.valid_to = TIMESTAMP '<delta_ts>'` to pick up only rows that THIS run closed, not rows a prior run closed at the same timestamp. The closed-at-delta-ts join key is what binds the INSERT to the just-fired MERGE.

</details>

**Wallet check.** About six cents so far. Each idempotency re-run adds one DELETE, one INSERT, one MERGE, and one INSERT. The checkpoint runs the re-apply itself, so the meter sees twice that on this checkpoint alone. Still well under a dime.

## Task 5: Point-in-time query. Return each customer's tier as of the seed timestamp

Back to the CRO's original question: "Show me revenue from customers who were in the Enterprise tier 90 days ago." With SCD Type 2 in place, the query is a single SELECT with a BETWEEN predicate on `valid_from` and `valid_to`. For any `:as_of` timestamp, every customer has exactly one row that satisfies `:as_of BETWEEN valid_from AND valid_to`, and that row's `tier` is the tier the customer held at that moment.

Run the point-in-time query at `:as_of = SEED_TS`. The result must match the legacy snapshot row-for-row: same customer_ids, same tiers. Customer 7 should show as Pro (the seed value), not Enterprise (where they were after delta 1) or the current Pro (after delta 2). Customer 13 should show as Free (the seed value), not Pro (the post-delta-1 value).

The future sentinel `TIMESTAMP '9999-12-31 00:00:00'` is what makes `BETWEEN` clean: every closed row has a real `valid_to`, every current row has the sentinel, and `BETWEEN` works symmetrically against both. NULL on `valid_to` would have forced `valid_to IS NULL OR :as_of <= valid_to` everywhere; the sentinel is the simpler shape.

In [ ]:
# NBVAL_SKIP
# Task 5: point-in-time SELECT at SEED_TS. Compare against the legacy
# snapshot via EXCEPT; the row-level diff must be zero in both directions.

# YOUR CODE: Run a SELECT customer_id, tier FROM {DB}.{TABLE_SCD2} WHERE
# TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to. Then compute two
# EXCEPT diffs: legacy MINUS as_of (rows that exist in legacy but not in
# the as_of snapshot) and as_of MINUS legacy (vice versa). Both must
# return zero. The fetch_scalar helper from the setup cell is fine for the
# count(*) wrapping each diff.

left_diff = int(fetch_scalar(f"""
    SELECT count(*) FROM (
        SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
        EXCEPT
        SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
        WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
    )
""") or 0)
right_diff = int(fetch_scalar(f"""
    SELECT count(*) FROM (
        SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
        WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
        EXCEPT
        SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
    )
""") or 0)

print(f"Rows in legacy but not in :as_of={SEED_TS}: {left_diff}")
print(f"Rows in :as_of={SEED_TS} but not in legacy: {right_diff}")

if left_diff == 0 and right_diff == 0:
    print(f"PASS: the SCD2 snapshot at :as_of={SEED_TS} reconstructs the legacy dim exactly.")
else:
    print("FAIL: the point-in-time slice disagrees with the legacy snapshot. Re-check the BETWEEN bounds and the sentinel on valid_to.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: point-in-time query at SEED_TS produces a row set that
# equals the legacy table row-for-row. Two EXCEPT diffs cover both
# directions; both must be zero.

def checkpoint_5(session):
    try:
        left = int(fetch_scalar(f"""
            SELECT count(*) FROM (
                SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
                EXCEPT
                SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
                WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
            )
        """) or 0)
        right = int(fetch_scalar(f"""
            SELECT count(*) FROM (
                SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
                WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
                EXCEPT
                SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
            )
        """) or 0)

        if left != 0 or right != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Point-in-time diff at :as_of={SEED_TS} is not zero. "
                    f"Legacy MINUS as_of: {left} rows; as_of MINUS legacy: {right} rows. "
                    f"Confirm valid_to on current rows is the future sentinel "
                    f"TIMESTAMP '{FUTURE_SENTINEL}' so BETWEEN catches them at any :as_of."
                ),
            )

        # Sanity check the touched customers show their seed tiers at SEED_TS,
        # not their post-delta tiers.
        rev_tier = fetch_scalar(
            f"SELECT tier FROM {DB}.{TABLE_SCD2} "
            f"WHERE customer_id = {CUSTOMER_REVERSION} "
            f"AND TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to"
        )
        if rev_tier != "Pro":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"At :as_of={SEED_TS}, customer {CUSTOMER_REVERSION} resolves to "
                    f"tier {rev_tier!r}, expected 'Pro' (their seed value). The "
                    f"point-in-time slice is leaking post-delta state into the snapshot."
                ),
            )

        upd_tier = fetch_scalar(
            f"SELECT tier FROM {DB}.{TABLE_SCD2} "
            f"WHERE customer_id = {CUSTOMER_UPDATE} "
            f"AND TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to"
        )
        if upd_tier != "Free":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"At :as_of={SEED_TS}, customer {CUSTOMER_UPDATE} resolves to "
                    f"tier {upd_tier!r}, expected 'Free' (their seed value)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

The point-in-time slice should reconstruct the legacy snapshot row-for-row. If the EXCEPT diff is nonzero, either a current row has the wrong `valid_to` (so `BETWEEN` does not catch it) or a historical row's `valid_from` is wrong (so `BETWEEN` catches a row that should not be in the slice).

</details>

<details><summary>Hint 2 (direction)</summary>

Run two EXCEPT diffs: legacy MINUS as_of, then as_of MINUS legacy. Each wrapped in `SELECT count(*) FROM (...)`. The query is `SELECT customer_id, tier FROM {TABLE_SCD2} WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to`. Both diffs must return zero. If only one side is nonzero, the bias tells you which side has the wrong shape (extra rows in as_of means double-current, missing rows in as_of means the sentinel was overwritten).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 5:

```python
left_diff = int(fetch_scalar(f"""
    SELECT count(*) FROM (
        SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
        EXCEPT
        SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
        WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
    )
""") or 0)
right_diff = int(fetch_scalar(f"""
    SELECT count(*) FROM (
        SELECT customer_id, tier FROM {DB}.{TABLE_SCD2}
        WHERE TIMESTAMP '{SEED_TS}' BETWEEN valid_from AND valid_to
        EXCEPT
        SELECT customer_id, tier FROM {DB}.{TABLE_LEGACY}
    )
""") or 0)
```

The two diffs are already in the task cell; the work in Task 5 is mostly confirming Tasks 1 through 4 landed correctly. If both diffs are nonzero, the most likely cause is a current row with `valid_to = NULL` instead of the future sentinel; re-run Task 2 (the initial-load MERGE) and Task 3 (both deltas) after confirming every INSERT writes `TIMESTAMP '9999-12-31 00:00:00'` on current rows.

</details>

**Wallet check.** About eight cents in total scan-wise. Five checkpoints, eleven Athena queries in the task cells, plus the checkpoint validators each run two to four scalar queries. Cleanup adds a handful more. The lab will land in the 30 to 60 cent range with re-runs and debugging.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order: Iceberg tables first (DROP TABLE removes both the Glue catalog
# entry and the Iceberg manifest pointers), then the Athena workgroup, then
# the Glue database, then the bucket. No critical (hourly-billed) resources
# in this lab.
#
# labrun-checks 0.8.0 now ships an iceberg_table adapter, so the _LabAdapter
# subclass below is functionally equivalent to the default AwsCleanupAdapter
# and is kept only for parity with this lab's prior version; a follow-up
# can drop it and call run_cleanup against the default adapter directly.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds iceberg_table support.

    Iceberg tables are dropped via Athena DDL because that removes both the
    Glue catalog entry AND the Iceberg manifest + data files in one call.
    A direct glue.delete_table would leave the Iceberg metadata orphaned in
    S3, and the next CREATE TABLE at the same LOCATION would fail with
    EXTERNAL_LOCATION_NOT_EMPTY.

    After the DROP, any straggler objects under the table's S3 prefix get
    swept by the s3_bucket adapter when the bucket itself is deleted.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _athena_client(self, credentials: dict):
        return boto3.client(
            "athena",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def _glue_client(self, credentials: dict):
        return boto3.client(
            "glue",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def _drop_iceberg(self, credentials: dict, resource) -> None:
        """Issue DROP TABLE IF EXISTS via Athena and poll until terminal."""
        athena_c = self._athena_client(credentials)
        parent = resource.parent or DB
        sql = f"DROP TABLE IF EXISTS {parent}.{resource.id}"
        try:
            start = athena_c.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": parent},
                WorkGroup=WG,
            )
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("InvalidRequestException", "ResourceNotFoundException"):
                # Workgroup is already gone (cleanup re-run); fall back to a
                # direct glue.delete_table so the catalog entry still goes.
                self._glue_client(credentials).delete_table(
                    DatabaseName=parent, Name=resource.id
                )
                return
            raise
        qid = start["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            resp = athena_c.get_query_execution(QueryExecutionId=qid)
            state = resp["QueryExecution"]["Status"]["State"]
            if state == "SUCCEEDED":
                return
            if state in ("FAILED", "CANCELLED"):
                reason = resp["QueryExecution"]["Status"].get("StateChangeReason", "")
                # If the table is already gone, fall through silently.
                if "does not exist" in reason.lower() or "not found" in reason.lower():
                    return
                raise RuntimeError(f"DROP TABLE {parent}.{resource.id} {state}: {reason}")
            time.sleep(1)
        raise RuntimeError(f"DROP TABLE {parent}.{resource.id} did not finish within 60 seconds.")

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "iceberg_table":
            self._drop_iceberg(credentials, resource)
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "iceberg_table":
            glue_c = self._glue_client(credentials)
            parent = resource.parent or DB
            try:
                glue_c.get_table(DatabaseName=parent, Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. Iceberg leaves
# metadata.json files, snapshot files, and data files; the s3_bucket adapter
# can only delete one page of objects at a time and Athena writes results
# under athena-results/ on top of the Iceberg writes.
print("Emptying the S3 bucket before teardown...")
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.05 to $0.40 based on the number of Athena queries actually executed.** Athena is the only metered service in this lab; Glue catalog operations and S3 storage on Iceberg metadata are effectively free at this volume. The bucket, database, workgroup, and three Iceberg tables were destroyed by the cleanup cell above, so nothing is still accruing. Check your AWS Billing console in 24 hours to confirm the exact number.

## These are not graded. They are for you.

Three questions worth sitting with before your next dimensional-modeling interview.

1. The lab uses `TIMESTAMP '9999-12-31 00:00:00'` as the future sentinel for `valid_to` on current rows. Why is that better than `valid_to = NULL`? Sketch the point-in-time SELECT under each shape and walk through what changes in the WHERE clause and the per-row evaluation cost. Which shape is friendlier to a downstream BI tool that auto-generates SQL?

2. SCD Type 2 with `is_current = true` carries a redundancy: at any moment, the row with `valid_to = future sentinel` IS the current row. Why keep the boolean? What query pattern gets simpler when `is_current` is present, and what happens to that pattern if you drop the column and force every query to derive currency from `valid_to`?

3. The lab does not cover deletes. When a customer closes their account, the canonical SCD2 move is to close the current row (set `valid_to` and `is_current = false`) but not insert a replacement. Walk through how the Task 3 MERGE+INSERT pattern would change to handle a delete signal in the source delta. What new column or predicate would the source need? What changes in the WHERE EXISTS clause in step 2?

## Exam-Style Practice

**Q1.** A customer in a Type 2 dimension goes from Pro to Enterprise on 2026-02-01, then from Enterprise back to Pro on 2026-03-01. The MERGE pipeline correctly closed each prior current row at the delta timestamp and inserted a fresh current row for each tier change. How many rows does the SCD2 table contain for this customer after the second delta lands?

A) 1, because SCD2 collapses reversions back to the original row.

B) 2, because the table tracks the current row and the immediately prior historical row.

C) 3, because every distinct period of validity gets its own row.

D) 4, because each tier transition produces both a closed row and a new current row.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because SCD2 does not collapse reversions; the whole point of Type 2 is that each distinct period of validity is preserved. Collapsing reversions is Type 1 (overwrite) or Type 3 (limited history).
- B) Wrong because the table tracks every historical period, not just the most recent one. With two transitions the customer has two closed rows and one current row, not one of each.
- C) Right because each period of validity is a separate row: original Pro (closed at delta 1), Enterprise (closed at delta 2), Pro current. This is the pattern Task 3 enforces with the two-statement MERGE+INSERT loop.
- D) Wrong because each transition produces one closed row (the prior current row) and one new current row, which together replace the single prior current row. Two transitions yield three rows, not four; the first transition's closed row and the new current row are net one extra row.

</details>

**Q2.** A data engineer's SCD Type 2 MERGE matches on `customer_id` alone (no `target.is_current = true` predicate) and uses `WHEN MATCHED THEN UPDATE SET valid_to = current_timestamp, is_current = false`. After the first delta, every customer in the SCD2 table has `is_current = false`. What went wrong?

A) The future sentinel was missing on the seed load; without it, BETWEEN evaluates the current rows as already closed.

B) The MERGE matched every existing row for each customer (including already-closed historical rows) and re-closed them all.

C) Athena Iceberg's MERGE silently rewrites every row when no `is_current` predicate is present; this is by design.

D) The `is_current` column needs a default value of `true` declared in the CREATE TABLE; without it, the MERGE writes NULL which evaluates as false.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because the future sentinel only matters for point-in-time queries via BETWEEN; the MERGE matches on a join key and predicate, not on a BETWEEN clause.
- B) Right because matching on `customer_id` alone means the MATCHED branch fires against every row for that customer, including rows already closed by earlier deltas. The fix is to add `AND target.is_current = true` to the ON clause (or to the MATCHED predicate) so only the current row is eligible for closure. This is the join key that Task 3 uses.
- C) Wrong because Athena Iceberg MERGE follows the SQL MERGE semantics; it does not have a hidden "rewrite everything" mode. The failure is in the join predicate, not the engine.
- D) Wrong because column defaults do not interact with MERGE this way; the MERGE explicitly UPDATEs `is_current` to false. Defaults would only matter on INSERT.

</details>

**Q3.** The downstream BI tool needs a point-in-time view that returns each customer's tier as of any user-supplied `:as_of` date. The SCD2 design uses `valid_from`, `valid_to`, and `is_current`, with `valid_to = TIMESTAMP '9999-12-31 00:00:00'` on every current row. Which query shape is the cleanest fit?

A) `SELECT customer_id, tier FROM customer_dim_scd2 WHERE is_current = true OR :as_of BETWEEN valid_from AND valid_to`

B) `SELECT customer_id, tier FROM customer_dim_scd2 WHERE :as_of BETWEEN valid_from AND valid_to`

C) `SELECT customer_id, tier FROM customer_dim_scd2 WHERE valid_from <= :as_of AND (valid_to IS NULL OR valid_to >= :as_of)`

D) `SELECT customer_id, tier FROM customer_dim_scd2 ORDER BY valid_from DESC LIMIT 1`

<details><summary>Show answer</summary>

**B**.

- A) Wrong because the OR returns every current row regardless of the `:as_of` parameter, which is wrong when `:as_of` is in the past (the current row is not the right row for a past snapshot). The query mixes "current" semantics with "as-of" semantics and breaks both.
- B) Right because the future sentinel on `valid_to` makes `BETWEEN` symmetric: it works for any `:as_of` from the seed date forward, including the present. This is the pattern Task 5 implements and is the canonical SCD2 point-in-time slice.
- C) Wrong only because the table does not use NULL on `valid_to`; the `IS NULL OR` branch is dead code. The query would still be correct in a design that uses NULL for current rows, but the sentinel design is cleaner and this Q tests recognition of the sentinel-friendly query.
- D) Wrong because ORDER BY + LIMIT 1 returns one row total (across all customers), not one per customer. To get one row per customer with ORDER BY, you need a window function or a self-join, both more complex than the BETWEEN shape.

</details>